
I###dentity Resolution Example using Vector Embeddings from a Doc2Vec MD.
 Siskmodel a Doc2Vec model
aDa
a,id C.v Sisk, 2026/0303/03-03-03v 2
is notebook requires a pre-trained doc2vec model, which you can download from here (quite large):
https://mega.nz/file/S2wxxArK#akoKdYl4SaO2JX29AHDWQ8skbDGnxrih9-mViK_ezWA

In [ ]:
#! pip install pandas
#! pip install scikit-learn
#! pip install gensim

In [1]:
import pandas as pd
import numpy as np
from gensim.models.doc2vec import Doc2Vec


In [2]:
df_inputdata = pd.read_csv('sample-data-messy_200.csv')

df_inputdata.shape

(200, 6)

In [3]:
# Calculate vector embeddings using a pretrained Doc2Vec model instead of an LLM
# Tokenize the text before calling the model

# Load the pretrained doc2vec model
doc2vec_model = Doc2Vec.load("doc2vec_wikipedia_dm.model")

# Function to generate embeddings for a row
from gensim.utils import simple_preprocess
def generate_doc2vec_embedding(row):
    # Exclude 'row_id' and 'true_id' columns and concatenate other columns
    text = " ".join(str(value) for key, value in row.items() if key not in ['row_id', 'true_id'])
    # Tokenize using gensim's simple_preprocess
    tokens = simple_preprocess(text)
    # Infer vector using the doc2vec model from tokenized text
    vector = doc2vec_model.infer_vector(tokens)
    return vector

# Apply the embedding function to each row
df_inputdata['embedding'] = df_inputdata.apply(generate_doc2vec_embedding, axis=1)

print("df_inputdata.shape:", df_inputdata.shape)

embedding_dimension = len(df_inputdata['embedding'].iloc[0])
print(f"The embedding dimension count is: {embedding_dimension}")

df_inputdata.shape: (200, 7)
The embedding dimension count is: 200


In [4]:
# L2-normalize vector embeddings and store in a new column
target_df = df if ("df" in globals() and "embedding" in df.columns) else df_inputdata

target_df["l2n_embedding"] = target_df["embedding"].apply(
    lambda v: v / np.linalg.norm(v) if np.linalg.norm(v) != 0 else v
)

In [5]:
# Using the inputdata above, construct a pairwise dataframe for every unique combination
from itertools import combinations

# Create lists of row_id and true_id
row_ids = df_inputdata['row_id'].tolist()
true_ids = df_inputdata['true_id'].tolist()

# Helper function to get field value, replacing NaN with empty string
def get_field(row, col):
    val = df_inputdata.iloc[row][col]
    return '' if pd.isna(val) else str(val)

# Create all pairwise combinations
pairwise_data = []
for i in range(len(df_inputdata)):
    for j in range(len(df_inputdata)):
        if i != j:  # Don't pair a row with itself
            # Concatenate name, email, address, and phone for both rows
            data1 = ' | '.join([
                get_field(i, 'name'),
                get_field(i, 'email'),
                get_field(i, 'address'),
                get_field(i, 'phone')
            ])
            data2 = ' | '.join([
                get_field(j, 'name'),
                get_field(j, 'email'),
                get_field(j, 'address'),
                get_field(j, 'phone')
            ])
            pairwise_data.append({
                'row_id1': row_ids[i],
                'true_id1': true_ids[i],
                'data1': data1,
                'embedding1': df_inputdata['l2n_embedding'].iloc[i],
                'row_id2': row_ids[j],
                'true_id2': true_ids[j],
                'data2': data2,
                'embedding2': df_inputdata['l2n_embedding'].iloc[j]
            })

df_pairwise = pd.DataFrame(pairwise_data)

df_pairwise.shape

(39800, 8)

In [6]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2
36622,row-0277,id-0002,Oilvia Smith | olivia.smith1@example.net | 235...,"[0.019182004, -0.09536417, -0.03474629, 0.0441...",row-0010,id-0076,Zeo Marshall | zoe.marshall5@example.net | 767...,"[0.11049928, 0.0011908276, 0.053378657, 0.0353..."
35619,row-0271,id-0037,"Nelson, Luke | lnelson@mail.example.org | | (...","[0.06969653, -0.022404011, -0.09231626, 0.0980...",row-0299,id-0005,Aav Miller | ava.miller4@example.net | 566 5 B...,"[-0.0014210455, -0.10720374, 0.003882196, 0.04..."
28008,row-0220,id-0021,"Lewis, Matthew | mlewis@mail.example.org | 212...","[0.12036761, -0.042241253, -0.029231189, 0.073...",row-0232,id-0018,"Robinson, Alexander | arobinson@mail.example.o...","[0.11782585, -0.07202989, 0.002475598, -0.0273..."


In [7]:
# For each pairwise row, calculate cosine similarity between embedding1 and
# embedding2, and store them in a new column called "cosine_similarity"
import numpy as np

def cosine_sim(v1, v2):
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    return float(np.dot(v1, v2) / denom) if denom != 0 else np.nan

df_pairwise["cosine_similarity"] = [
    cosine_sim(v1, v2)
    for v1, v2 in zip(df_pairwise["embedding1"], df_pairwise["embedding2"])
]


In [8]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity
14496,row-0120,id-0087,"Newman, Sadie | snewman@mail.example.org | | ...","[0.069973536, -0.0460641, -0.06949448, 0.07888...",row-0257,id-0051,Cnonor Morris | connor.morris1@example.net | ...,"[-0.020206874, -0.09794406, 0.0458508, 0.00512...",0.606863
31459,row-0243,id-0074,"Hale, Victoria | vhale@mail.example.org | 7474...","[0.05641072, -0.02268944, 0.016180085, 0.04310...",row-0026,id-0045,"Jonathan Parker | | 45 North St, Springfield,...","[0.10732134, -0.05193026, 0.024807611, 0.09840...",0.487368
11000,row-0092,id-0090,Vviian Grant | vivian.grant5@example.net | 909...,"[0.02698844, -0.08022822, -0.010777297, 0.0247...",row-0093,id-0070,"Ford, Wesley | wford@mail.example.org | 7070 7...","[0.11819961, -0.014445235, -0.02196393, -0.015...",0.205584


In [9]:
# Min-max normalize cosine similarity to a 0-100 scale and store that as the score
min_sim = df_pairwise["cosine_similarity"].min()
max_sim = df_pairwise["cosine_similarity"].max()

if max_sim == min_sim:
    df_pairwise["score"] = 100.0
else:
    df_pairwise["score"] = (
        (df_pairwise["cosine_similarity"] - min_sim) / (max_sim - min_sim) * 100
    )

# round score to 2 decimal places
df_pairwise["score"] = df_pairwise["score"].round(2)

In [10]:
df_pairwise.sample(3)

,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
17764,row-0154,id-0030,Lvei Lopez | levi.lopez1@example.net | 3031 30...,"[0.021875512, -0.01675754, -0.0057270443, 0.03...",row-0089,id-0100,"Hopkins, Denise | dhopkins@mail.example.org | ...","[0.08734754, -0.06708574, -0.003583559, 0.0509...",0.395130,54.05
30901,row-0239,id-0039,"Mitchell, Caleb | cmitchell@mail.example.org |...","[0.105417006, -0.0058697225, 0.014390119, 0.05...",row-0093,id-0070,"Ford, Wesley | wford@mail.example.org | 7070 7...","[0.11819961, -0.014445235, -0.02196393, -0.015...",0.605900,71.71
16241,row-0142,id-0026,"Andrew Young | | 26 Glen Ave., Springfield, I...","[0.09661099, -0.02112836, -0.05364477, 0.07942...",row-0200,id-0019,Mcihael Clark | michael.clark4@example.net | 1...,"[-0.013436336, -0.008483671, 0.009799965, 0.00...",0.455785,59.13


In [11]:
# Display rows where true_id1 equals true_id2
df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]


,row_id1,true_id1,data1,embedding1,row_id2,true_id2,data2,embedding2,cosine_similarity,score
142,row-0002,id-0080,"Chavez, Leah | lchavez@mail.example.org | 8080...","[0.074444205, -0.054377615, -0.012208029, 0.04...",row-0224,id-0080,Laeh Chavez | leah.chavez2@example.net | 8081 ...,"[-0.018432124, -0.06319761, 0.013705014, 0.028...",0.653979,75.74
313,row-0003,id-0013,"James Harris | | 13 Sycamore St, Springfield,...","[0.06979027, -0.042792052, -0.019873396, 0.130...",row-0190,id-0013,Jmaes Harris | james.harris5@example.net | 131...,"[0.041132215, -0.031976517, -0.01047454, 0.099...",0.658685,76.13
486,row-0004,id-0030,"Lopez, Levi | llopez@mail.example.org | 3030 3...","[0.10874024, -0.030590288, 0.030725153, 0.0181...",row-0154,id-0030,Lvei Lopez | levi.lopez1@example.net | 3031 30...,"[0.021875512, -0.01675754, -0.0057270443, 0.03...",0.682671,78.14
647,row-0005,id-0011,"Jackson, Henry | hjackson@mail.example.org | 1...","[0.06755011, -0.05408241, -0.09449815, -0.0034...",row-0087,id-0011,"Henry Jackson | | 11 Willow Dr, Springfield, ...","[0.056184985, -0.022838587, -0.11915596, 0.075...",0.665801,76.73
868,row-0008,id-0041,Rayn Roberts | ryan.roberts5@example.net | | ...,"[-0.006110252, -0.04575712, 0.0019261658, 0.05...",row-0121,id-0041,"Roberts, Ryan | rroberts@mail.example.org | 41...","[0.11845905, 0.016274268, 0.07266901, 0.035066...",0.458675,59.38
...,...,...,...,...,...,...,...,...,...,...
38954,row-0294,id-0018,"Alexander Robinson | | 18 Cypress Ave., Sprin...","[0.07445288, -0.063098945, -0.03804755, 0.0550...",row-0232,id-0018,"Robinson, Alexander | arobinson@mail.example.o...","[0.11782585, -0.07202989, 0.002475598, -0.0273...",0.589523,70.34
39159,row-0296,id-0039,Claeb Mitchell | caleb.mitchell3@example.net |...,"[0.051291812, -0.01822941, 0.036799453, 0.0595...",row-0239,id-0039,"Mitchell, Caleb | cmitchell@mail.example.org |...","[0.105417006, -0.0058697225, 0.014390119, 0.05...",0.670221,77.10
39261,row-0298,id-0047,"Edwards, Thomas | tedwards@mail.example.org | ...","[0.013324999, -0.049867325, -0.06489167, 0.098...",row-0096,id-0047,Tohmas Edwards | thomas.edwards4@example.net |...,"[-0.006368457, -0.054412644, 0.0022343304, 0.0...",0.480124,61.17
39531,row-0299,id-0005,Aav Miller | ava.miller4@example.net | 566 5 B...,"[-0.0014210455, -0.10720374, 0.003882196, 0.04...",row-0207,id-0005,"Miller, Ava | amiller@mail.example.org | 567 5...","[0.076022975, -0.09001074, -0.011028007, 0.014...",0.654050,75.74


In [12]:
# Examine the score spread where they were true matches

# Filter rows where true_id1 equals true_id2
matches = df_pairwise[df_pairwise['true_id1'] == df_pairwise['true_id2']]
true_match_count = len(matches)

# Calculate statistics
count = len(matches)
min_score = matches['score'].min()
avg_score = matches['score'].mean()
max_score = matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 200
Min Score: 40.75
Avg Score: 70.98
Max Score: 89.1


In [13]:
# Examine the score spread where they are NOT true matches

# Filter rows where true_id1 does NOT equal true_id2
non_matches = df_pairwise[df_pairwise['true_id1'] != df_pairwise['true_id2']]
non_match_count = len(non_matches)

# Calculate statistics
count = len(non_matches)
min_score = non_matches['score'].min()
avg_score = non_matches['score'].mean()
max_score = non_matches['score'].max()

print(f"Count: {count}")
print(f"Min Score: {min_score}")
print(f"Avg Score: {avg_score:.2f}")
print(f"Max Score: {max_score}")

Count: 39600
Min Score: 0.0
Avg Score: 62.85
Max Score: 100.0


In [14]:
# Choose the matching score cutoff threshold that gets the most acceptable mix 
# of false positives and false negatives
#cutoff = 47.76  # This is the middle of the average scores...reasonable starting point
cutoff = 65.00
df_pairwise["match"] = (df_pairwise["score"] >= cutoff).astype(int)

# Calculate Precision & Recall as our accuracy metrics
# True positives: predicted match (1) and actually same person (true_id1 == true_id2)
true_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# False positives: predicted match (1) but actually different people (true_id1 != true_id2)
false_positives = df_pairwise[
    (df_pairwise["match"] == 1) & (df_pairwise["true_id1"] != df_pairwise["true_id2"])
]

# False negatives: predicted no match (0) but actually same person (true_id1 == true_id2)
false_negatives = df_pairwise[
    (df_pairwise["match"] == 0) & (df_pairwise["true_id1"] == df_pairwise["true_id2"])
]

# True negatives: predicted no match (0) and actually different people (true_id1 != true_id2)
true_negatives = df_pairwise[
    (df_pairwise["match"] == 0) & (df_pairwise["true_id1"] != df_pairwise["true_id2"])
]


# Calculate Precision & Recall, plus F1 score
precision = len(true_positives) / (len(true_positives) + len(false_positives))
recall = len(true_positives) / (len(true_positives) + len(false_negatives))
F1_score = 2 * (precision * recall) / (precision + recall)

print("Non-LLM example...")
print("Generated true matches in sample data:", true_match_count)
print("Predicted matches:", (df_pairwise["match"] == 1).sum())
print("Generated true non-matches in sample data:", non_match_count)
print("Predicted non-matches:", (df_pairwise["match"] == 0).sum())
print("")
print(f"TRUE POSITIVES: {len(true_positives)}")
print(f"FALSE POSITIVES: {len(false_positives)}")
print(f"FALSE NEGATIVES: {len(false_negatives)}")
print(f"TRUE NEGATIVES: {len(true_negatives)}")
print("Check: TRUE positives + FALSE negatives = 200? ->", len(true_positives) + len(false_negatives))
print("")
print(f"PRECISION: {precision:.4f}  (What % of predicted matches were correct true matches?)")
print(f"RECALL: {recall:.4f}  (What % of true matches were correctly predicted?)")
print(f"F1 SCORE: {F1_score:.4f}  (Harmonic mean of precision and recall)") 

Non-LLM example...
Generated true matches in sample data: 200
Predicted matches: 18004
Generated true non-matches in sample data: 39600
Predicted non-matches: 21796

TRUE POSITIVES: 154
FALSE POSITIVES: 17850
FALSE NEGATIVES: 46
TRUE NEGATIVES: 21750
Check: TRUE positives + FALSE negatives = 200? -> 200

PRECISION: 0.0086  (What % of predicted matches were correct true matches?)
RECALL: 0.7700  (What % of true matches were correctly predicted?)
F1 SCORE: 0.0169  (Harmonic mean of precision and recall)
